In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
PROC = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROC / "cleaned_apple_sales_enriched_realistic.csv")

In [3]:
df.head()

,sale_id,sale_date,store_id,product_id,quantity,product_name,launch_date,price,store_name,city,...,price_realistic,days_from_start,product_trend,trend_factor,store_factor,mu_demand,quantity_realistic,sales_amount_realistic,month,launch_year
0,RW-18212,2020-01-01,ST-41,P-73,2,Apple One,2020-01-01,199,Apple Central World,Bangkok,...,199.0,0,0.000783,1.0,1.219421,1.039677,3,597.0,1,2020
1,RX-6848,2020-01-01,ST-31,P-40,7,iPhone SE (3rd Generation),2020-01-01,429,Apple Shinjuku,Tokyo,...,429.0,0,0.000154,1.0,1.172288,10.813781,6,2574.0,1,2020
2,SV-39198,2020-01-01,ST-9,P-73,7,Apple One,2020-01-01,199,Apple Park Visitor Center,Cupertino,...,199.0,0,0.000783,1.0,0.958735,6.406862,5,995.0,1,2020
3,ZI-218670,2020-01-01,ST-20,P-70,1,Apple Fitness+,2020-01-01,79,Apple Kaerntner Strasse,Vienna,...,79.0,0,0.000692,1.0,1.109330,0.795094,1,79.0,1,2020
4,VB-48589,2020-01-01,ST-60,P-18,9,Beats Solo Pro,2020-01-01,299,Apple Antara,Mexico City,...,299.0,0,0.000495,1.0,1.203289,4.415985,4,1196.0,1,2020


In [4]:
print(df.columns)

Index(['sale_id', 'sale_date', 'store_id', 'product_id', 'quantity',
       'product_name', 'launch_date', 'price', 'store_name', 'city',
       'category_id', 'category_name', 'sales_amount', 'invalid_launch_flag',
       'product_age_days', 'exchange_rate', 'inflation_rate',
       'internet_usage_pct', 'country_norm_mapped', 'year', 'gdp_type',
       'gdp_per_capita', 'season_factor', 'economic_factor', 'promo_flag',
       'promo_factor', 'price_realistic', 'days_from_start', 'product_trend',
       'trend_factor', 'store_factor', 'mu_demand', 'quantity_realistic',
       'sales_amount_realistic', 'month', 'launch_year'],
      dtype='object')


In [5]:
df.shape

(1040200, 36)

In [6]:
unique_products = df["product_name"].unique()

print(unique_products)
print(len(unique_products))

['Apple One' 'iPhone SE (3rd Generation)' 'Apple Fitness+'
 'Beats Solo Pro' 'Beats Studio Buds' 'iPhone 14 Pro Max' 'iMac 24-inch'
 'Magic Trackpad' 'Apple Watch Nike Edition' 'MagSafe Battery Pack'
 'HomePod mini' 'AirTag' 'MacBook Air (Retina)' 'iPhone 12' 'iMac Pro'
 'Lightning to USB Cable' 'iPad Pro (M2)' 'MacBook Pro 13-inch'
 'Apple Watch Series 6' 'iPad Pro 12.9-inch' 'AirPods (3rd Generation)'
 'Apple Watch Series 9' 'Leather Case for iPhone'
 'Magic Keyboard with Touch ID' 'Apple Pencil (2nd Generation)'
 'MacBook Pro 14-inch' 'Smart Keyboard Folio' 'Mac Studio'
 'iPad (10th Generation)' 'Apple Watch Hermès' 'Apple News+'
 'Silicone Case for iPhone' 'Apple Music' 'Mac Pro (2023)'
 'Apple Watch Ultra' 'Magic Mouse' 'AirPods Pro' 'iPhone 14 Plus'
 'iMac 27-inch' 'Apple Watch Series 7' 'iPad Pro (M1)' 'Magic Keyboard'
 'iPhone 13 Pro' 'iPad Air (4th Generation)' 'MacBook Air (M2)'
 'AirPods Pro (2nd Generation)' 'MacBook Pro (Touch Bar)' 'iPhone 12 mini'
 'HomePod' 'AirPods Max

In [7]:
# لو الداتا الأصلية فيها الأعمدة product_name و launch_date
df[["product_name", "launch_date"]].drop_duplicates().sort_values("launch_date")

,product_name,launch_date
0,Apple One,2020-01-01
107,iCloud,2020-01-01
106,AirPods (2nd Generation),2020-01-01
105,MacBook Pro 16-inch,2020-01-01
102,Smart Cover for iPad,2020-01-01
...,...,...
995966,Apple Watch Ultra,2024-08-30
1004177,Apple Arcade,2024-09-13
1017778,AirPods Pro,2024-10-06
1018391,iPhone 14 Pro Max,2024-10-07


In [8]:
pd.set_option("display.max_rows", None)

product_launches = (
    df[["product_name", "launch_date"]]
    .drop_duplicates()
    .sort_values("launch_date")
)

print(product_launches)

                          product_name launch_date
0                            Apple One  2020-01-01
107                             iCloud  2020-01-01
106           AirPods (2nd Generation)  2020-01-01
105                MacBook Pro 16-inch  2020-01-01
102               Smart Cover for iPad  2020-01-01
95                            Mac Mini  2020-01-01
89          iPad mini (6th Generation)  2020-01-01
88            HomePod (2nd Generation)  2020-01-01
86                      Mac Pro (Rack)  2020-01-01
109                        Apple TV HD  2020-01-01
85                   iPhone 12 Pro Max  2020-01-01
83                           iPhone 13  2020-01-01
78                       Mac Mini (M2)  2020-01-01
77            iMac with Retina Display  2020-01-01
74                Apple Watch Series 8  2020-01-01
73                         AirPods Max  2020-01-01
70                             HomePod  2020-01-01
69                      iPhone 12 mini  2020-01-01
67             MacBook Pro (Tou

In [9]:
# Group by category and count the number of UNIQUE product names
unique_products_per_category = df.groupby('category_name')['product_name'].nunique().reset_index()

# Rename the columns for clarity
unique_products_per_category.columns = ['category_name', 'total_unique_products']

# Sort from highest number of products to lowest
unique_products_per_category = unique_products_per_category.sort_values(by='total_unique_products', ascending=False)

display(unique_products_per_category)


,category_name,total_unique_products
0,Accessories,14
5,Smartphone,13
1,Audio,11
2,Desktop,10
3,Laptop,10
8,Tablet,10
9,Wearable,9
7,Subscription Service,7
6,Streaming Device,3
4,Smart Speaker,2


In [10]:
# Group by both year and category, then count the UNIQUE product names sold
unique_products_per_year_cat = df.groupby(['year', 'category_name'])['product_name'].nunique().reset_index()

# Rename the columns for clarity
unique_products_per_year_cat.columns = ['year', 'category_name', 'total_unique_products']

# Create a pivot table to make it easy to read (Categories on the left, Years across the top)
pivot_unique = unique_products_per_year_cat.pivot(index='category_name', columns='year', values='total_unique_products')

# Fill any missing data with 0 and convert to integers
pivot_unique = pivot_unique.fillna(0).astype(int)

display(pivot_unique)


year,2020,2021,2022,2023,2024
category_name,,,,,
Accessories,14,14,14,14,14
Audio,11,11,11,11,11
Desktop,10,10,10,10,10
Laptop,10,10,10,10,10
Smart Speaker,2,2,2,2,2
Smartphone,13,13,13,13,13
Streaming Device,3,3,3,3,3
Subscription Service,7,7,7,7,7
Tablet,10,10,10,10,10


In [11]:
# Group by both the Launch Year and category, then count the UNIQUE product names
unique_products_per_launch = df.groupby(['launch_year', 'category_name'])['product_name'].nunique().reset_index()

# Rename the columns for clarity
unique_products_per_launch.columns = ['launch_year', 'category_name', 'total_unique_products']

# Create a pivot table (Categories on the left, Launch Years across the top)
pivot_launch = unique_products_per_launch.pivot(index='category_name', columns='launch_year', values='total_unique_products')

# Fill any missing data with 0 and convert to integers
pivot_launch = pivot_launch.fillna(0).astype(int)

display(pivot_launch)


launch_year,2020,2021,2022,2023,2024
category_name,,,,,
Accessories,14,0,2,4,4
Audio,11,2,4,0,3
Desktop,10,3,3,1,1
Laptop,10,2,1,3,2
Smart Speaker,2,0,0,1,0
Smartphone,13,4,1,2,3
Streaming Device,3,1,0,1,0
Subscription Service,7,2,1,0,2
Tablet,10,4,2,3,0
